# Notebook 1 - Download 13F Filings from SEC

so this is where everything starts. the SEC has a free API that gives you all the filing data for any fund manager. you just need their CIK number which is basically their unique ID on EDGAR.

i started with berkshire hathaway because i am a big fan of warren buffett but the code works for literally any fund manager, just change the CIK in config.py

what this notebook does:
- hits the SEC submissions API with the manager CIK
- finds all 13F filings across the last N quarters  
- for each filing, figures out which XML file is the actual holdings table (this was trickier than i expected)
- downloads and caches the XML so we dont have to hit the API again
- saves everything to filings.parquet

one thing i learned the hard way - the SEC filing directory has multiple XML files and only one of them has the actual holdings. the other one is just a cover page. so we have to check the content of each file to figure out which one we actually want

In [1]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

working directory: /content/13F-Analytics


In [2]:
# install if running on colab
# !pip install pyarrow requests pandas matplotlib numpy -q

# set your name and email - SEC rejects requests without this
# !sed -i 's/Your Name your_email@example.com/Your Actual Name youremail@email.com/' src/config.py

In [3]:
from src import config
from src.utils import make_session
from src.sec import build_filings_dataset

print("manager:", config.MANAGER_NAME)
print("CIK:", config.CIK)
print("quarters to fetch:", config.N_QUARTERS)
print("user agent:", config.USER_AGENT)

manager: Berkshire Hathaway
CIK: 0001067983
quarters to fetch: 8
user agent: Karanveer Singh klnu5@asu.edu


In [4]:
# this will take a few minutes, its downloading 8 quarters of XML from SEC
session = make_session()
filings = build_filings_dataset(session, n_quarters=config.N_QUARTERS)
filings

,accessionNumber,form,filingDate,reportDate,quarter,primaryDocument,xml_file,xml_url,local_xml_path
0,0001193125-26-226661,13F-HR,2026-05-15,2026-03-31,2026Q1,xslForm13F_X02/primary_doc.xml,53405.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2026Q1_000...
1,0001193125-26-054580,13F-HR,2026-02-17,2025-12-31,2025Q4,xslForm13F_X02/primary_doc.xml,50240.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2025Q4_000...
2,0001193125-25-282901,13F-HR,2025-11-14,2025-09-30,2025Q3,xslForm13F_X02/primary_doc.xml,46994.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2025Q3_000...
3,0000950123-25-008343,13F-HR,2025-08-14,2025-06-30,2025Q2,xslForm13F_X02/primary_doc.xml,43977.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2025Q2_000...
4,0000950123-25-008361,13F-HR/A,2025-08-14,2025-03-31,2025Q1,xslForm13F_X02/primary_doc.xml,43981.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2025Q1_000...
5,0000950123-25-002701,13F-HR,2025-02-14,2024-12-31,2024Q4,xslForm13F_X02/primary_doc.xml,39042.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2024Q4_000...
6,0000950123-24-011775,13F-HR,2024-11-14,2024-09-30,2024Q3,xslForm13F_X02/primary_doc.xml,36917.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2024Q3_000...
7,0000950123-24-008740,13F-HR,2024-08-14,2024-06-30,2024Q2,xslForm13F_X02/primary_doc.xml,34725.xml,https://www.sec.gov/Archives/edgar/data/106798...,/content/13F-Analytics/data/raw_xml/2024Q2_000...


In [5]:
# make sure every quarter has a verified XML file
# if any show None here something went wrong with that filing
filings[["quarter", "form", "filingDate", "reportDate", "xml_file"]]

,quarter,form,filingDate,reportDate,xml_file
0,2026Q1,13F-HR,2026-05-15,2026-03-31,53405.xml
1,2025Q4,13F-HR,2026-02-17,2025-12-31,50240.xml
2,2025Q3,13F-HR,2025-11-14,2025-09-30,46994.xml
3,2025Q2,13F-HR,2025-08-14,2025-06-30,43977.xml
4,2025Q1,13F-HR/A,2025-08-14,2025-03-31,43981.xml
5,2024Q4,13F-HR,2025-02-14,2024-12-31,39042.xml
6,2024Q3,13F-HR,2024-11-14,2024-09-30,36917.xml
7,2024Q2,13F-HR,2024-08-14,2024-06-30,34725.xml


outputs saved to data/processed/filings.parquet and XML cached in data/raw_xml/ - next run notebook 2